In [1]:
import requests
import pandas as pd

# ==========================================
# FASE 1: DESCOBERTA DINÂMICA DE METADADOS
# ==========================================
def gerar_mapa_ufs() -> dict:
    """
    Consome a API do IBGE para mapear dynamicamente as siglas das UFs para seus respectivos IDs.
    Retorna um dicionário no formato {'SP': 35, 'RJ': 33, ...}.
    """
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        estados = r.json()
        # Dictionary Comprehension para busca O(1)
        return {est['sigla']: est['id'] for est in estados}
    except Exception as e:
        print(f"Erro ao mapear UFs: {e}")
        return {}

# Inicializa o mapeamento
mapa_ufs = gerar_mapa_ufs()


# ==========================================
# FASE 2: INGESTÃO DE MUNICÍPIOS POR SIGLA
# ==========================================
def capturar_municipios(sigla: str, mapa: dict):
    """
    Valida a sigla recebida contra o mapa dinâmico antes de fazer a requisição.
    Retorna a lista de municípios em formato JSON/dicionário.
    """
    sigla_formatada = sigla.upper()
    uf_id = mapa.get(sigla_formatada)

    # Validação semântica de entrada
    if not uf_id:
        return f"Sigla de UF '{sigla}' não encontrada."

    url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{uf_id}/municipios"
    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.RequestException as e:
        return f"Erro de conexão: {e}"


# ==========================================
# FASE 3: TRANSFORMAÇÃO E ANÁLISE COM PANDAS
# ==========================================
def processar_dados_municipios(dados_json: list):
    """
    Converte o JSON retornado em DataFrame Pandas e executa as análises requisitadas.
    """
    if isinstance(dados_json, str):
        print(f"Não foi possível processar: {dados_json}")
        return None

    # Normalização dos dados aninhados da API do IBGE
    df = pd.json_normalize(dados_json)

    # 1. Total de municípios
    total_municipios = len(df)
    print(f"Total de municípios no estado: {total_municipios}")

    # 2. Filtrar municípios com nomes extensos (mais de 12 caracteres)
    df['tamanho_nome'] = df['nome'].str.len()
    df_nomes_longos = df[df['tamanho_nome'] > 12]
    print(f"Municípios com mais de 12 caracteres: {len(df_nomes_longos)}")

    # Meta Analítica: Distribuição de municípios por microrregião
    if 'microrregiao.nome' in df.columns:
        distribuicao_microrregiao = df['microrregiao.nome'].value_counts()
        print("\n--- Distribuição por Microrregião ---")
        print(distribuicao_microrregiao.head())

    # 3. Exportar dados limpos para CSV
    df[['id', 'nome', 'microrregiao.nome']].to_csv("municipios_limpos.csv", index=False)
    print("\nDados limpos exportados para 'municipios_limpos.csv'.")

    return df


# Execução do fluxo completo para teste (Exemplo com SP)
dados_sp = capturar_municipios("SP", mapa_ufs)
df_sp = processar_dados_municipios(dados_sp)


# ==========================================
# FASE 4: TESTES DE RESILIÊNCIA
# ==========================================
print("\n=== Executando Testes de Resiliência ===")

# Cenário 1: Timeout curto (0.01s)
try:
    requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/estados", timeout=0.01)
except requests.exceptions.Timeout:
    print("[OK] Cenário Timeout: Exceção capturada com sucesso.")

# Cenário 2: URL Inválida (Erro 404)
try:
    r_invalida = requests.get("https://servicodados.ibge.gov.br/api/v1/localidades/endpoint_inexistente")
    r_invalida.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(f"[OK] Cenário URL Inválida: HTTP Error capturado ({e}).")

# Cenário 3: Entrada Inválida ("XX")
resultado_xx = capturar_municipios("XX", mapa_ufs)
print(f"[OK] Cenário Entrada Inválida: {resultado_xx}")

Total de municípios no estado: 645
Municípios com mais de 12 caracteres: 182

--- Distribuição por Microrregião ---
microrregiao.nome
Presidente Prudente      30
São José do Rio Preto    29
Jales                    23
Bauru                    21
Birigui                  18
Name: count, dtype: int64

Dados limpos exportados para 'municipios_limpos.csv'.

=== Executando Testes de Resiliência ===
[OK] Cenário Timeout: Exceção capturada com sucesso.
[OK] Cenário URL Inválida: HTTP Error capturado (404 Client Error: Not Found for url: https://servicodados.ibge.gov.br/api/v1/localidades/endpoint_inexistente).
[OK] Cenário Entrada Inválida: Sigla de UF 'XX' não encontrada.


## Respostas para a Discussão Teórica

1. *Por que a utilização de um dicionário dinâmico é superior a uma lista estática de códigos de UF?*

O dicionário dinâmico garante que o software seja resiliente a alterações de infraestrutura ou de cadastros (como alteração de códigos IBGE ou criação de novos territórios). Além disso, a estrutura de dicionário (dict) fornece complexidade de busca $O(1)$, garantindo máxima eficiência no mapeamento em tempo de execução.
#-----------------------------------------------------------------------------
2. *Como a validação de entrada (camada de software) economiza recursos de rede e processamento?*

Validar a entrada na camada de software evita o envio de requisições HTTP desnecessárias (e propensas ao erro 404) para o servidor da API. Isso reduz a latência da aplicação, consome menos banda de rede e evita sobrecarregar os servidores do IBGE com chamadas inválidas.
#-------------------------------------------------------------------------------
3. *De que forma a interoperabilidade via APIs permite que softwares financeiros se mantenham atualizados com dados governamentais?*

A integração via APIs RESTful conecta diretamente o sistema financeiro com a fonte oficial da verdade (IBGE). Assim que o órgão governamental atualiza dados de territórios, microrregiões ou estatísticas territoriais, a fintech consome as atualizações instantaneamente em tempo de execução, sem necessidade de atualizar bases locais estáticas ou efetuar novos deploys do código-fonte.